# Notebook 2. All runs together (MNIST)

**About this copy.** This is the MNIST copy of `model_fitting/2_all_runs.ipynb` on the zain branch. The questions, methods and figures are the same. The changes are listed below.

- The initial scale `s` (column `init_scale`) takes the place of `alpha`. Every column, interaction and held-out test that used `alpha` uses `s`.
- A run has grokked when its test accuracy reaches 0.80, in place of 0.95. The time-varying Cox model uses the levels 0.70, 0.80 and 0.85 in place of 0.80, 0.90 and 0.95.
- The step budget is 500,000. The code reads it from the data.
- The original drops weight decay 3e-3 by hand. Here the code drops every weight decay level at which no run memorises, and prints which levels it dropped.
- The landmark is the kernel step nearest `M.LANDMARK_TARGET`, in place of step 2,430.

The notebook reads `mnist/sweep/` through `mnist_settings.py`. Set `MNIST_SWEEP_DATA` to read another folder. The results of the original do not carry over, so every takeaway is left to be written once the sweep has run.

**The question.** This is the same question as notebook 1: can `A_t`, `S_t` and `R_t`, read early in training, predict when a run groks? This time we pool every weight decay level at which some run memorises. Notebook 1 has one cell per mix of `s` and width in each of its two groups, which is 16 cells on the full grid. Pooling the levels gives the models much more to learn from.

### Words you need

| Word | What it means |
| --- | --- |
| step | One update of the weights. Each run gets 500,000 steps, which is the **budget**. |
| grokked | The test accuracy reached 0.80. The **grokking step** `T` is the first step where that happens, counted from step 0. The report (§3.5) defines `T` the same way at level 0.95, which MNIST does not reach reliably. |
| censored | The run never reached 0.80 within the budget. We keep it, and we know only that `T` is over 500,000. |
| memorised | The training accuracy reached 0.99. |
| cell | One mix of initial scale `s`, width `N` and weight decay `ηλ` (column `eta_kappa`). Each cell has 5 seeds. |
| `s` | The initial scale. The sweep multiplies the initial weights by `s`. It plays the part of `alpha` in the modular addition sweep and sets how lazy the network starts. |
| kernel | A table that says how much a training step on one input moves the output on another. We use the report's kernel of the summed outputs (Eq. 5), here on a fixed draw of 1,000 MNIST test images. |
| `S_t` | How much bigger the kernel is than at step 0. 1 means the same size. |
| `R_t` | How much the kernel has turned since step 0. 0 means the same shape. |
| `A_t` | How well the kernel lines up with the labels. Higher means inputs with the same answer look more alike. |
| landmark | The kernel step nearest `M.LANDMARK_TARGET`. The setup prints it. We read the kernel there, before any run groks. |

Black marks all runs. Violet, teal and gold mark `A_t`, `S_t` and `R_t`.

**The split.** Each seed draws its own 1,000 training images with data seed 42 plus the seed number. The test set is all 10,000 MNIST test images.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import chi2
from lifelines import CoxPHFitter, CoxTimeVaryingFitter, KaplanMeierFitter, LogNormalAFTFitter
from lifelines.utils import concordance_index
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

import mnist_settings as M

plt.style.use('fitting.mplstyle')
GREY, INK, SURFACE = '#8a8984', '#0b0b0b', '#fcfcfb'
KERNEL_COLOURS = {'A_t': '#7b5cd6', 'S_t': '#1a9e8f', 'R_t': '#c99a06'}
kernel = ['A_t', 'S_t', 'R_t']
INPUTS = ['settings', 'kernel', 'both']
KNOB, S = M.KNOB, M.KNOB_LABEL      # the column init_scale, and its name s in labels


def tag(level):
    """The column name of the first step at which test accuracy reaches level, such as t_gen80."""
    return f't_gen{round(level * 100)}'


GEN = tag(M.LEVEL)                         # the grokking level, test accuracy 0.80
LEVEL_COLS = [tag(level) for level in M.LEVELS]

## Load the data

This is the same loading step as notebook 1. `M.load` also returns the landmark.

In [ ]:
runs, acc, kern, LANDMARK = M.load()
BUDGET = int(runs['last_step'].max())

# The first step at which each accuracy reaches each level. NaN means it never did.
events = runs.set_index('run_id')
events['t_mem99'] = acc[acc['train_acc'] >= M.MEM].groupby('run_id')['step'].min()
for level in sorted(set(M.LEVELS) | {M.LEVEL}):
    events[tag(level)] = acc[acc['test_acc'] >= level].groupby('run_id')['step'].min()
events = events.reset_index()

events['grokked'] = events[GEN].notna().astype(int)          # 1 = grokked, 0 = censored
events['T'] = events[GEN].fillna(events['last_step'])        # the grokking step, or the budget if censored
events['cell'] = events.apply(M.cell_name, axis=1)

# Drop every weight decay level at which no run memorises. Part 1 shows the outcomes at each level.
memorises = events.groupby('eta_kappa')['t_mem99'].apply(lambda t: t.notna().any())
dropped = memorises.index[~memorises].tolist()
print('landmark step:', LANDMARK, '  budget:', BUDGET)
print('weight decay levels dropped because no run memorises:', [f'{w:g}' for w in dropped] or 'none')

data = events.merge(kern[kern['step'] == LANDMARK].drop(columns='step'), on='run_id')
pooled = data[~data['eta_kappa'].isin(dropped)].copy()
pd.Series({'runs': len(pooled), 'cells': pooled['cell'].nunique(), 'grokked': pooled['grokked'].sum()})

## Part 1. The runs at a glance

The sweep runs every mix of 4 values of `s`, 4 widths, 7 weight decay levels and 5 seeds, which makes 560 runs on the full grid. Here is what happens to the runs in the data at each weight decay level.

In [ ]:
events['outcome'] = np.select([events['grokked'] == 1, events['t_mem99'].notna()],
                              ['grokked', 'memorised, did not grok'], default='never memorised')
outcome = (pd.crosstab(events['eta_kappa'], events['outcome'])
           .reindex(columns=['grokked', 'memorised, did not grok', 'never memorised'], fill_value=0))
outcome.index = [f'{w:g}' for w in outcome.index]
per_level = events.groupby('eta_kappa').size()

ax = outcome.plot.bar(stacked=True, color=[INK, GREY, '#d9d8d3'], rot=0, figsize=(8, 4))
ax.set_xlabel('Weight decay ηλ')
ax.set_ylabel(f'Runs ({per_level.iloc[0]} per level)' if per_level.nunique() == 1 else 'Runs')
ax.set_title('What happens to the runs at each weight decay level')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()
outcome

**Takeaway.** To be written once the MNIST sweep has run.

## Part 2. How long does grokking take at each level?

A **Kaplan-Meier curve** shows the share of runs that haven't grokked yet at each step. A censored run counts as "not yet" for the whole budget, so nothing is thrown away (report Eq. 9). The report draws one curve per cell. Here we pool the cells of each level to see the big picture. The levels at which no run memorises are left out, as the setup prints.

In [ ]:
seen_T = pooled.loc[pooled['grokked'] == 1, 'T']
left = 10 ** np.floor(np.log10(seen_T.min())) if len(seen_T) else LANDMARK   # a round step below the first grokking

fig, ax = plt.subplots(figsize=(8, 4.5))
shades = plt.cm.Greys(np.linspace(0.35, 1, pooled['eta_kappa'].nunique()))
for shade, (level, g) in zip(shades, pooled.groupby('eta_kappa')):
    km = KaplanMeierFitter().fit(g['T'], g['grokked'])
    km.survival_function_.plot(ax=ax, drawstyle='steps-post', color=shade, legend=False)
    ax.text(BUDGET, km.survival_function_.iloc[-1, 0], f' {level:g}', va='center', fontsize=8, color=shade)
ax.set_xscale('log')
ax.set_xlim(left, BUDGET)
ax.axhline(0.5, color=GREY, linestyle=':')
ax.set_xlabel('Step')
ax.set_ylabel('Share of runs not grokked yet')
ax.set_title('One curve per weight decay level (labels on the right)')
plt.tight_layout()
plt.show()

Where a curve crosses the dotted line at one half, we can read off the median grokking step. The next picture gives that median for every mix of `s` and weight decay.

In [ ]:
rows = []
for (a, w), g in pooled.groupby([KNOB, 'eta_kappa']):
    km = KaplanMeierFitter().fit(g['T'], g['grokked'])
    rows.append({KNOB: a, 'eta_kappa': w, 'median T': km.median_survival_time_})
medians = pd.DataFrame(rows).replace(np.inf, np.nan)
levels = sorted(pooled['eta_kappa'].unique())
scales = sorted(pooled[KNOB].unique())

fig, ax = plt.subplots(figsize=(8, 4))
for k, a in enumerate(scales):
    m = medians[medians[KNOB] == a]
    ax.plot([levels.index(w) for w in m['eta_kappa']], m['median T'], marker='os^Dvp'[k % 6],
            linestyle=['-', '--', ':', '-.'][k % 4], color=INK, markersize=8, label=f'{S} = {a:g}')
ax.set_xticks(range(len(levels)), [f'{w:g}' for w in levels])
ax.set_yscale('log')
ax.set_xlabel('Weight decay ηλ')
ax.set_ylabel('Median grokking step')
ax.set_title('A missing point means fewer than half the runs grok')
ax.legend()
plt.show()
medians.pivot(index=KNOB, columns='eta_kappa', values='median T')

**Takeaway.** To be written once the MNIST sweep has run.

## Part 3. No peeking

A model may only use what is known before a run groks. We read the kernel at the landmark, which the setup prints, and check that no run has reached any of the test levels by then. The time-varying Cox model in Part 8 starts at the landmark at every level in `M.LEVELS`, so the check covers all of them.

In [ ]:
earliest = events[LEVEL_COLS].min()
for col, step in earliest.dropna().items():
    assert step > LANDMARK, (f'A run reached test accuracy 0.{col[-2:]} at step {step:.0f}, which is not after the landmark '
                             f'at step {LANDMARK}. Lower M.LANDMARK_TARGET in mnist_settings.py and run again.')

bins = np.logspace(1, np.log10(BUDGET), 40)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col, title in [(axes[0], 't_mem99', 'Step where the run memorised'),
                       (axes[1], GEN, 'Step where the run grokked')]:
    ax.hist(pooled[col].dropna(), bins=bins, color=INK, alpha=0.7)
    ax.axvline(LANDMARK, color=INK, linestyle='--')
    ax.text(LANDMARK, 0.95, ' landmark', transform=ax.get_xaxis_transform(), va='top')
    ax.set_xscale('log')
    ax.set_xlabel('Step')
    ax.set_title(title)
axes[0].set_ylabel('Runs')
plt.tight_layout()
plt.show()

seen = pooled[pooled['grokked'] == 1]
log_T = np.log10(seen['T'])
pd.Series({**{f'earliest step at test accuracy 0.{col[-2:]}': earliest[col] for col in LEVEL_COLS},
           'highest test accuracy at the landmark': acc.loc[acc['step'] == LANDMARK, 'test_acc'].max(),
           'share of the spread in log T between cells': log_T.groupby(seen['cell']).transform('mean').var() / log_T.var()}).round(3)

**Takeaway.** To be written once the MNIST sweep has run.

Every score below holds out whole cells, so seeds of one cell never sit on both sides of a split.

## Part 4. The ladder of models

We try four models, from the simplest up. Each gets three sets of inputs.

- **settings**: `s`, width and weight decay, as yes/no columns. One column per value lets each value have its own effect.
- **kernel**: `A_t`, `S_t` and `R_t` at the landmark.
- **both**: settings and kernel together.

If the kernel tells us something the settings don't, **both** should beat **settings**. Every score is on **held-out cells**: we fit on every cell but one and predict the one left out, for each cell in turn.

| Step | Model | Predicts | Runs it uses | Score |
| --- | --- | --- | --- | --- |
| 1 | Linear regression | log `T` | grokked runs only | concordance |
| 2 | Logistic regression | groks within the budget, yes or no | all | AUC |
| 3 | AFT | log `T` | all, with censored runs as "over 500,000" | concordance |
| 4 | Cox | the chance of grokking at the next step | all | concordance |

**Concordance** takes every pair of runs where we know which grokked first, and counts how often the model orders them right. **AUC** is the chance that a run that grokked gets a higher score than one that didn't. For both, 0.5 is a coin toss and 1 is perfect.

The yes/no columns look like this. The smallest value of each setting has no column, because it is the baseline.

In [ ]:
settings_cols = pooled[[KNOB, 'width', 'eta_kappa']].rename(columns={KNOB: S})
dummies = pd.get_dummies(settings_cols.astype('category'), drop_first=True, dtype=float)
dummies.head()

In [ ]:
d = pooled.sort_values('T').reset_index(drop=True)
seen_T = d['grokked'] == 1
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(d.index[seen_T], d.loc[seen_T, 'T'], color=INK, s=8, label='grokked: we know T')
ax.scatter(d.index[~seen_T], d.loc[~seen_T, 'T'], color=GREY, marker='^', s=20, label=f'never grokked: T is over {BUDGET:,}')
ax.set_yscale('log')
ax.set_xlabel('Runs, sorted by T')
ax.set_ylabel('Grokking step T')
ax.set_title('Censored runs only give us a lower limit for T')
ax.legend(loc='lower right')
plt.show()

## Part 5. Step 1: linear regression

The simplest model is a straight line through log `T`. It needs a real `T`, so it **drops every run that never grokked**. `scores` collects every held-out score in this notebook.

In [ ]:
scores = {}
g = pooled[pooled['grokked'] == 1].copy()
g_dummies = dummies.loc[g.index]
for cell in g['cell'].unique():
    train, test = g['cell'] != cell, g['cell'] == cell
    z = (g[kernel] - g.loc[train, kernel].mean()) / g.loc[train, kernel].std()
    for inputs, X in [('settings', g_dummies), ('kernel', z), ('both', g_dummies.join(z))]:
        line = LinearRegression().fit(X[train], np.log10(g.loc[train, 'T']))
        g.loc[test, f'lin_{inputs}'] = line.predict(X[test])
for inputs in INPUTS:
    scores[('linear regression', inputs)] = concordance_index(g['T'], g[f'lin_{inputs}'])
print(f'{len(g)} grokked runs are used and {len(pooled) - len(g)} runs that never grokked are dropped.')
pd.Series({i: scores[('linear regression', i)] for i in INPUTS}, name='held-out concordance').round(3)

**Takeaway.** To be written once the MNIST sweep has run.

## Part 6. Step 2: logistic regression

Next we keep every run but ask a simpler question: **will it grok within the budget, yes or no?** Logistic regression gives each run a probability of grokking.

In [ ]:
for cell in pooled['cell'].unique():
    train, test = pooled['cell'] != cell, pooled['cell'] == cell
    z = (pooled[kernel] - pooled.loc[train, kernel].mean()) / pooled.loc[train, kernel].std()
    for inputs, X in [('settings', dummies), ('kernel', z), ('both', dummies.join(z))]:
        clf = LogisticRegression(max_iter=1000).fit(X[train], pooled.loc[train, 'grokked'])
        pooled.loc[test, f'prob_{inputs}'] = clf.predict_proba(X[test])[:, 1]

rows = {}
for inputs in INPUTS:
    scores[('logistic regression', inputs)] = roc_auc_score(pooled['grokked'], pooled[f'prob_{inputs}'])
    rows[inputs] = {'AUC': scores[('logistic regression', inputs)],
                    'balanced accuracy': balanced_accuracy_score(pooled['grokked'], pooled[f'prob_{inputs}'] > 0.5),
                    'accuracy': ((pooled[f'prob_{inputs}'] > 0.5) == pooled['grokked']).mean()}
pd.DataFrame(rows).T.round(3)

**Balanced accuracy** averages the hit rate on grokked runs and on the others, so always guessing one answer scores 0.5. A **confusion matrix** counts right and wrong guesses with a cut at probability 0.5.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, inputs in zip(axes, INPUTS):
    guess = (pooled[f'prob_{inputs}'] > 0.5).astype(int)
    matrix = pd.crosstab(pooled['grokked'], guess).reindex(index=[0, 1], columns=[0, 1], fill_value=0).to_numpy()
    ax.imshow(matrix, cmap=LinearSegmentedColormap.from_list('n', [SURFACE, GREY]), vmin=0, vmax=len(pooled))
    for i in range(2):
        for j in range(2):
            ax.text(j, i, matrix[i, j], ha='center', va='center', fontsize=12)
    ax.set_xticks([0, 1], ['predicted no', 'predicted yes'])
    ax.set_yticks([0, 1], ['did not grok', 'grokked'])
    ax.set_title(inputs)
    ax.grid(False)
fig.suptitle('Held-out guesses of the logistic model')
plt.tight_layout()
plt.show()

Where do the mistakes fall? Here is the share of runs guessed right, by setting, for **settings** against **both**.

In [ ]:
for inputs in ['settings', 'both']:
    pooled[f'right_{inputs}'] = (pooled[f'prob_{inputs}'] > 0.5) == pooled['grokked']

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, col in zip(axes, ['eta_kappa', KNOB, 'width']):
    right = pooled.groupby(col)[['right_settings', 'right_both']].mean()
    right.index = [f'{v:g}' for v in right.index]
    right.plot.bar(ax=ax, color=[GREY, INK], rot=0, ylim=(0.5, 1), legend=False)
    ax.set_xlabel({'eta_kappa': 'weight decay ηλ', KNOB: f'initial scale {S}', 'width': 'width N'}[col])
axes[0].set_ylabel('Share guessed right')
axes[2].legend(['settings', 'both'], loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 7. Step 3: the AFT model

Linear regression drops the runs that never grok. Logistic regression keeps them but forgets the timing. The **accelerated failure time (AFT)** model does both jobs (report Eq. 10 and 11).

It's a linear regression on log `T` with one upgrade. When a run grokked, we know `T`, and the model fits it like normal regression. When it didn't, we only know `T` is over 500,000, and the model uses exactly that: it asks for a line that makes "over 500,000" likely for that run. Each input stretches or squeezes the time. A coefficient `a` multiplies the median grokking step by e^a, so a negative `a` means sooner.

A tiny penalty (0.01) keeps the fit stable when a setting has few grokked runs. The loop can take a few minutes.

In [ ]:
for cell in pooled['cell'].unique():
    train, test = pooled['cell'] != cell, pooled['cell'] == cell
    z = (pooled[kernel] - pooled.loc[train, kernel].mean()) / pooled.loc[train, kernel].std()
    for inputs, X in [('settings', dummies), ('kernel', z), ('both', dummies.join(z))]:
        aft = LogNormalAFTFitter(penalizer=0.01).fit(X[train].assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
        pooled.loc[test, f'aft_{inputs}'] = aft.predict_median(X[test]).to_numpy().ravel()
for inputs in INPUTS:
    scores[('AFT', inputs)] = concordance_index(pooled['T'], pooled[f'aft_{inputs}'], pooled['grokked'])
pd.Series({i: scores[('AFT', i)] for i in INPUTS}, name='held-out concordance').round(3)

**Takeaway.** To be written once the MNIST sweep has run.

### Which kernel numbers matter in the AFT?

We fit the **both** model on every run and read its three kernel coefficients. The **likelihood ratio test** asks whether the three together improve on the settings alone.

In [ ]:
z = (pooled[kernel] - pooled[kernel].mean()) / pooled[kernel].std()
with_kernel = LogNormalAFTFitter(penalizer=0.01).fit(dummies.join(z).assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
without = LogNormalAFTFitter(penalizer=0.01).fit(dummies.assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
lr_p = chi2.sf(2 * (with_kernel.log_likelihood_ - without.log_likelihood_), 3)
coefs = with_kernel.summary.loc['mu_'].loc[kernel, ['coef', 'coef lower 95%', 'coef upper 95%', 'p']]
coefs['times the median'] = np.exp(coefs['coef'])

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.errorbar(coefs['coef'], range(3), xerr=[coefs['coef'] - coefs['coef lower 95%'], coefs['coef upper 95%'] - coefs['coef']],
            fmt='o', markersize=8, color=INK)
ax.axvline(0, color=GREY)
ax.set_yticks(range(3), kernel)
ax.invert_yaxis()
ax.set_xlabel('AFT coefficient (left of 0 = groks sooner)')
ax.set_title(f'All runs, joint p = {lr_p:.2g}')
plt.tight_layout()
plt.show()
coefs.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

These intervals count each seed as an independent run, so they are a bit too narrow.

## Part 8. Step 4: the Cox model

The Cox model doesn't predict a time. At every step it looks at the runs that haven't grokked yet and asks which one is most likely to grok next. That chance is the **hazard**. Each input multiplies the hazard by e^γ. So a **positive** γ means sooner, which is the opposite sign to the AFT (report Eq. 13 and 14).

We use it three ways.

- **(a)** With the kernel at the landmark, scored on held-out cells like the other models.
- **(b)** The report's own version, where the kernel changes along the run.
- **(c)** An extra that isn't in the report. It compares only the seeds of one cell, so the settings drop out.

### (a) The kernel at the landmark

In [ ]:
for cell in pooled['cell'].unique():
    train, test = pooled['cell'] != cell, pooled['cell'] == cell
    z = (pooled[kernel] - pooled.loc[train, kernel].mean()) / pooled.loc[train, kernel].std()
    for inputs, X in [('settings', dummies), ('kernel', z), ('both', dummies.join(z))]:
        cox = CoxPHFitter(penalizer=0.01).fit(X[train].assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
        pooled.loc[test, f'cox_{inputs}'] = cox.predict_partial_hazard(X[test]).to_numpy().ravel()
for inputs in INPUTS:
    # a higher hazard means sooner, so we flip its sign before scoring
    scores[('Cox', inputs)] = concordance_index(pooled['T'], -pooled[f'cox_{inputs}'], pooled['grokked'])
pd.Series({i: scores[('Cox', i)] for i in INPUTS}, name='held-out concordance').round(3)

**Takeaway.** To be written once the MNIST sweep has run.

### (b) The report's model: the kernel at every step

Here the kernel can change along the run (report Eq. 13). We cut each run into stretches between the saved kernel steps. Each stretch carries the kernel from its start, so the model never sees the future. The report calls this left-continuous. A stretch gets `event` = 1 if the run groks inside it. We start at the landmark and fit each level in `M.LEVELS`: 0.70, 0.80 and 0.85. They replace the report's 0.80, 0.90 and 0.95, which MNIST does not reach reliably.

In [ ]:
long = kern.merge(pooled[['run_id', 'last_step'] + LEVEL_COLS], on='run_id').sort_values(['run_id', 'step'])
long['start'] = long['step']
long['next'] = long.groupby('run_id')['step'].shift(-1)     # the next saved step of the same run
long = long.dropna(subset=['next'])

tv_rows, tv_fits = {}, {}
for level in LEVEL_COLS:
    lf = long.copy()
    lf['grokked'] = lf[level].notna().astype(int)
    lf['T'] = lf[level].fillna(lf['last_step'])
    lf = lf[(lf['start'] >= LANDMARK) & (lf['start'] < lf['T'])]      # from the landmark until the run groks
    lf['stop'] = np.minimum(lf['next'], lf['T'])
    lf['event'] = ((lf['stop'] == lf['T']) & (lf['grokked'] == 1)).astype(int)
    assert lf['event'].sum() == lf.groupby('run_id')['grokked'].first().sum()   # one event per grokked run
    at_lm = lf[lf['start'] == LANDMARK]
    lf[kernel] = (lf[kernel] - at_lm[kernel].mean()) / at_lm[kernel].std()
    ctv = CoxTimeVaryingFitter().fit(lf[['run_id', 'start', 'stop', 'event'] + kernel], id_col='run_id',
                                     event_col='event', start_col='start', stop_col='stop')
    tv_fits[level] = ctv.summary
    tv_rows[level] = {**ctv.summary['coef'].to_dict(), **ctv.summary['p'].add_prefix('p ').to_dict(),
                      'joint p': ctv.log_likelihood_ratio_test().p_value}

lf[['run_id', 'start', 'stop', 'event'] + kernel].head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
shades = plt.cm.Greys(np.linspace(0.35, 1, len(LEVEL_COLS)))
offsets = 0.2 * (np.arange(len(LEVEL_COLS)) - (len(LEVEL_COLS) - 1) / 2)
for level, shade, offset in zip(LEVEL_COLS, shades, offsets):
    s = tv_fits[level]
    ax.errorbar(s['coef'], np.arange(3) + offset, xerr=[s['coef'] - s['coef lower 95%'], s['coef upper 95%'] - s['coef']],
                fmt='o', color=shade, label=f'test accuracy 0.{level[-2:]}')
ax.axvline(0, color=GREY)
ax.set_yticks(range(3), kernel)
ax.invert_yaxis()
ax.set_xlabel('Cox coefficient (right of 0 = groks sooner)')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()
pd.DataFrame(tv_rows).T.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

As in notebook 1, this model reads the kernel up to the step before grokking. So it describes the run while it groks and gives no early warning. The report notes (§3.5) that these inputs come from the run itself, so the model says nothing about cause.

### (c) Extra: inside one cell

This Cox model gets one **stratum** per cell. Each cell has its own baseline, and the model only compares seeds of the same cell. So the settings drop out completely.

In [ ]:
z = (pooled[kernel] - pooled[kernel].mean()) / pooled[kernel].std()
cox = CoxPHFitter().fit(pooled[['T', 'grokked', 'cell']].join(z), 'T', 'grokked', strata='cell', robust=True)
print('joint p', round(cox.log_likelihood_ratio_test().p_value, 4))
cox.summary[['coef', 'coef lower 95%', 'coef upper 95%', 'p']].round(3)

The same idea as a picture, for the runs that grokked. Each dot is one seed. Its position shows how far its `A_t` and its grokking step sit above or below the average of its cell-mates.

In [ ]:
g = pooled[pooled['grokked'] == 1].copy()
g = g[g.groupby('cell')['T'].transform('size') >= 2]
z = (g['A_t'] - pooled['A_t'].mean()) / pooled['A_t'].std()
x = z - z.groupby(g['cell']).transform('mean')
y = np.log10(g['T']) - np.log10(g['T']).groupby(g['cell']).transform('mean')
slope, intercept = np.polyfit(x, y, 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, color=KERNEL_COLOURS['A_t'], s=14, alpha=0.6)
ax.plot(np.sort(x), intercept + slope * np.sort(x), color=INK)
ax.axhline(0, color=GREY, linewidth=1)
ax.axvline(0, color=GREY, linewidth=1)
ax.set_xlabel('A_t above its cell-mates (standard deviations)')
ax.set_ylabel('log10 T above its cell-mates')
ax.set_title(f'r = {np.corrcoef(x, y)[0, 1]:.2f}, {len(g)} runs in {g["cell"].nunique()} cells')
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 9. The four models side by side

Every held-out score from Parts 5 to 8 in one picture. Logistic regression is scored by AUC and the others by concordance. For both, 0.5 is a coin toss.

In [ ]:
models = ['linear regression', 'logistic regression', 'AFT', 'Cox']
table = pd.Series(scores).unstack().reindex(index=models, columns=INPUTS)
ax = table.plot.bar(rot=0, color=['#c4c3bd', GREY, INK], ylim=(0.4, 1), figsize=(9, 3.8), legend=False)
ax.axhline(0.5, color=GREY, linestyle=':')
ax.set_ylabel('Held-out score')
ax.legend(ax.containers, INPUTS, loc='lower right')
plt.show()
table.round(3)

### Which of `A_t`, `S_t` and `R_t` predicts?

Now each number goes in on its own, in every model. **Alone** shows everything the number knows, including what it knows about the settings. **On top of the settings** shows what it adds beyond them. A held-out score a little below 0.5 means the number carries almost no signal. This loop fits a few thousand models and takes a few minutes.

In [ ]:
one = {}
grok = pooled['grokked'] == 1
for number in kernel:
    for how in ['alone', 'on top of the settings']:
        pred = pd.DataFrame(index=pooled.index, columns=models, dtype=float)
        for cell in pooled['cell'].unique():
            train, test = pooled['cell'] != cell, pooled['cell'] == cell
            z = (pooled[[number]] - pooled.loc[train, [number]].mean()) / pooled.loc[train, [number]].std()
            X = z if how == 'alone' else dummies.join(z)
            fit_on = X[train].assign(T=pooled['T'], grokked=pooled['grokked'])
            if (test & grok).any():
                line = LinearRegression().fit(X[train & grok], np.log10(pooled.loc[train & grok, 'T']))
                pred.loc[test & grok, 'linear regression'] = line.predict(X[test & grok])
            clf = LogisticRegression(max_iter=1000).fit(X[train], pooled.loc[train, 'grokked'])
            pred.loc[test, 'logistic regression'] = clf.predict_proba(X[test])[:, 1]
            aft = LogNormalAFTFitter(penalizer=0.01).fit(fit_on, 'T', 'grokked')
            pred.loc[test, 'AFT'] = aft.predict_median(X[test]).to_numpy().ravel()
            cox = CoxPHFitter(penalizer=0.01).fit(fit_on, 'T', 'grokked')
            pred.loc[test, 'Cox'] = -cox.predict_partial_hazard(X[test]).to_numpy().ravel()
        one[(how, number)] = {
            'linear regression': concordance_index(pooled.loc[grok, 'T'], pred.loc[grok, 'linear regression']),
            'logistic regression': roc_auc_score(pooled['grokked'], pred['logistic regression']),
            'AFT': concordance_index(pooled['T'], pred['AFT'], pooled['grokked']),
            'Cox': concordance_index(pooled['T'], pred['Cox'], pooled['grokked'])}
one = pd.DataFrame(one).T.sort_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, how in zip(axes, ['alone', 'on top of the settings']):
    m = one.loc[how].T.reindex(index=models, columns=kernel)
    ax.imshow(m, cmap=LinearSegmentedColormap.from_list('s', [SURFACE, INK]), vmin=0.5, vmax=1, aspect='auto')
    for r in range(m.shape[0]):
        for c in range(m.shape[1]):
            v = m.iloc[r, c]
            ax.text(c, r, f'{v:.2f}', ha='center', va='center', color='white' if v > 0.8 else INK)
    ax.set_xticks(range(3), kernel)
    ax.set_yticks(range(len(models)), models)
    ax.set_title(how)
    ax.grid(False)
fig.suptitle('Held-out score of one kernel number at a time (0.5 = coin toss)')
plt.tight_layout()
plt.show()
one.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

## Part 10. What `s` does

This part looks at the initial scale `s` on its own.

### Where runs grok, by `s` and weight decay

In [ ]:
share = pooled.pivot_table(index=KNOB, columns='eta_kappa', values='grokked', aggfunc='mean')
share.columns = [f'{w:g}' for w in share.columns]
fig, ax = plt.subplots(figsize=(8, 0.6 * len(share) + 1.2))
ax.imshow(share, cmap=LinearSegmentedColormap.from_list('share', [SURFACE, INK]), vmin=0, vmax=1)
for i in range(share.shape[0]):
    for j in range(share.shape[1]):
        v = share.iloc[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', color='white' if v > 0.6 else INK)
ax.set_xticks(range(share.shape[1]), share.columns)
ax.set_yticks(range(share.shape[0]), [f'{a:g}' for a in share.index])
ax.set_xlabel('Weight decay ηλ')
ax.set_ylabel(f'initial scale {S}')
ax.set_title('Share of runs that grok')
ax.grid(False)
plt.tight_layout()
plt.show()

### How much `s` and width stretch the grokking step

We fit the settings-only AFT on every run. Then e^a turns each coefficient into "times the median grokking step", compared with the smallest `s` or the smallest width at the same other settings.

In [ ]:
settings_fit = LogNormalAFTFitter(penalizer=0.01).fit(dummies.assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
scale_width = [c for c in dummies if c.startswith((f'{S}_', 'width_'))]
multipliers = np.exp(settings_fit.summary.loc['mu_'].loc[scale_width, ['coef', 'coef lower 95%', 'coef upper 95%']])
multipliers.columns = ['times the median', 'lower 95%', 'upper 95%']

fig, ax = plt.subplots(figsize=(7, 3.2))
m = multipliers
ax.errorbar(m['times the median'], range(len(m)), xerr=[m['times the median'] - m['lower 95%'], m['upper 95%'] - m['times the median']],
            fmt='o', color=INK)
ax.axvline(1, color=GREY)
ax.set_xscale('log')
ax.set_yticks(range(len(m)), m.index)
ax.invert_yaxis()
ax.set_xlabel('Times the median grokking step (1 = no change)')
plt.tight_layout()
plt.show()
multipliers.round(2)

**Takeaway.** To be written once the MNIST sweep has run.

The intervals count seeds as independent, so they are too narrow.

### The kernel at the landmark for each `s`

In [ ]:
scales = sorted(pooled[KNOB].unique())
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, number in zip(axes, kernel):
    groups = [pooled.loc[pooled[KNOB] == a, number] for a in scales]
    box = ax.boxplot(groups, tick_labels=[f'{a:g}' for a in scales], patch_artist=True, widths=0.5)
    for patch in box['boxes']:
        patch.set_facecolor(KERNEL_COLOURS[number])
        patch.set_alpha(0.6)
    ax.set_xlabel(f'initial scale {S}')
    ax.set_title(number)
plt.tight_layout()
plt.show()
pooled.groupby(KNOB)[kernel].median().round(3)

**Takeaway.** To be written once the MNIST sweep has run.

### Does the kernel still help inside one value of `s`?

Each value of `s` has one cell per mix of width and weight decay. We repeat the held-out AFT and logistic tests inside each value, with width and weight decay as the settings. A value of `s` in which every run or no run grokked cannot be scored, and its scores are left empty.

In [ ]:
rows = {}
for a in sorted(pooled[KNOB].unique()):
    df = pooled[pooled[KNOB] == a].copy()
    label = f'{S} = {a:g}'
    if df['grokked'].nunique() < 2:
        rows[label] = {'cells with a grokked run': df.loc[df['grokked'] == 1, 'cell'].nunique()}
        continue
    d_a = pd.get_dummies(df[['width', 'eta_kappa']].astype('category'), drop_first=True, dtype=float)
    for cell in df['cell'].unique():
        train, test = df['cell'] != cell, df['cell'] == cell
        z = (df[kernel] - df.loc[train, kernel].mean()) / df.loc[train, kernel].std()
        for inputs, X in [('settings', d_a), ('both', d_a.join(z))]:
            aft = LogNormalAFTFitter(penalizer=0.01).fit(X[train].assign(T=df['T'], grokked=df['grokked']), 'T', 'grokked')
            df.loc[test, f'a_{inputs}'] = aft.predict_median(X[test]).to_numpy().ravel()
            clf = LogisticRegression(max_iter=1000).fit(X[train], df.loc[train, 'grokked'])
            df.loc[test, f'p_{inputs}'] = clf.predict_proba(X[test])[:, 1]
    rows[label] = {'cells with a grokked run': df.loc[df['grokked'] == 1, 'cell'].nunique(),
                   'AFT concordance, settings': concordance_index(df['T'], df['a_settings'], df['grokked']),
                   'AFT concordance, both': concordance_index(df['T'], df['a_both'], df['grokked']),
                   'AUC, settings': roc_auc_score(df['grokked'], df['p_settings']),
                   'AUC, both': roc_auc_score(df['grokked'], df['p_both'])}
pd.DataFrame(rows).T.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

### Is the kernel standing in for mixes of `s` and weight decay?

The settings model gives `s` and weight decay one effect each. It can't say that one value of `s` groks only in a narrow band of weight decay. Maybe the kernel helps because it carries that. So we give the settings one column per mix of `s` and weight decay, and check whether the kernel still adds anything.

In [ ]:
pooled['s_wd'] = pooled[KNOB].map('{:g}'.format) + '_' + pooled['eta_kappa'].map('{:g}'.format)
combos = pd.get_dummies(pooled[['s_wd', 'width']].astype('category'), drop_first=True, dtype=float)
for cell in pooled['cell'].unique():
    train, test = pooled['cell'] != cell, pooled['cell'] == cell
    z = (pooled[kernel] - pooled.loc[train, kernel].mean()) / pooled.loc[train, kernel].std()
    for inputs, X in [('combos', combos), ('combos_kernel', combos.join(z))]:
        aft = LogNormalAFTFitter(penalizer=0.01).fit(X[train].assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
        pooled.loc[test, f'aft_{inputs}'] = aft.predict_median(X[test]).to_numpy().ravel()
        clf = LogisticRegression(max_iter=1000).fit(X[train], pooled.loc[train, 'grokked'])
        pooled.loc[test, f'prob_{inputs}'] = clf.predict_proba(X[test])[:, 1]

rows = {}
for inputs, name in [('settings', 'settings'), ('both', 'settings + kernel'),
                     ('combos', f'mixes of {S} and weight decay'), ('combos_kernel', 'mixes + kernel')]:
    rows[name] = {'AFT concordance': concordance_index(pooled['T'], pooled[f'aft_{inputs}'], pooled['grokked']),
                  'logistic AUC': roc_auc_score(pooled['grokked'], pooled[f'prob_{inputs}'])}
pd.DataFrame(rows).T.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

### Can a model predict a value of `s` it has never seen?

We hold out one value of `s`, train on the others, and predict it. The models get no `s` column, so they have to work out its effect from what they see. A held-out value at which every run or no run grokked adds no pairs to the AUC, so the scores pool the predictions over all held-out values.

In [ ]:
width_wd = pd.get_dummies(pooled[['width', 'eta_kappa']].astype('category'), drop_first=True, dtype=float)
rows = {}
for name, use_settings, use_kernel in [('width + weight decay', True, False), ('kernel only', False, True),
                                       ('width + weight decay + kernel', True, True)]:
    pred = pd.Series(index=pooled.index, dtype=float)
    prob = pd.Series(index=pooled.index, dtype=float)
    for a in pooled[KNOB].unique():
        train, test = pooled[KNOB] != a, pooled[KNOB] == a
        X = pd.DataFrame(index=pooled.index)
        if use_settings:
            X = X.join(width_wd)
        if use_kernel:
            X = X.join((pooled[kernel] - pooled.loc[train, kernel].mean()) / pooled.loc[train, kernel].std())
        aft = LogNormalAFTFitter(penalizer=0.01).fit(X[train].assign(T=pooled['T'], grokked=pooled['grokked']), 'T', 'grokked')
        pred[test] = aft.predict_median(X[test]).to_numpy().ravel()
        prob[test] = LogisticRegression(max_iter=1000).fit(X[train], pooled.loc[train, 'grokked']).predict_proba(X[test])[:, 1]
    rows[name] = {'AFT concordance': concordance_index(pooled['T'], pred, pooled['grokked']),
                  'logistic AUC': roc_auc_score(pooled['grokked'], prob)}
held_scale = pd.DataFrame(rows).T

ax = held_scale.plot.bar(rot=0, color=[GREY, INK], ylim=(0.4, 1), figsize=(8, 3.5))
ax.axhline(0.5, color=GREY, linestyle=':')
ax.set_ylabel(f'Score on the held-out {S}')
plt.show()
held_scale.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

## Summary

**Takeaway.** To be written once the MNIST sweep has run.